## Keys — Primary, Foreign, Natural, and Surrogate

Keys are the columns (or combinations of columns) that **identify and connect** rows. Understanding them is essential for joins, deduplication, and maintaining data integrity.

### Types of key

| Key type | Definition | Example |
| --- | --- | --- |
| **Primary key** | A column (or set of columns) that **uniquely identifies** every row in a table. No nulls, no duplicates. | A system-generated `id` column, or a composite of `pupil_id` + `school_urn` |
| **Natural key** | A key derived from **real-world attributes** rather than a system-generated value. Meaningful to humans. | `national_insurance_number`, `urn`, `uln` |
| **Surrogate key** | A **system-generated identifier** with no real-world meaning, used purely to uniquely identify rows. Typically an auto-incrementing integer or UUID. | An `id` column populated by `ROW_NUMBER()` or `UUID()` |
| **Foreign key** | A column in one table that **references the primary key** of another table, creating a relationship between them. | `school_urn` in a pupil table referencing `urn` in a school table |

> **Note:** These categories are not mutually exclusive. A **natural key can also be the primary key** — and often is when the table has no surrogate. For example, if a table's grain is one row per pupil per school, the composite natural key `(pupil_id, school_urn)` is also the primary key. The distinction is about *origin* (real-world attribute vs system-generated) rather than *role* (uniquely identifying rows).

### How keys relate to cardinality

The **primary key defines the grain**. If your primary key is `(pupil_id, school_urn)`, then one row = one pupil at one school. If you find two rows with the same primary key values, you have a duplicate.

### Natural keys vs surrogate keys

Natural keys are intuitive but can be **unstable** — a school’s URN might change after an academy conversion, or a pupil’s name might be updated. Surrogate keys are stable but meaningless to humans. Many systems use both: a surrogate key for internal relationships and natural keys for human-readable lookups.

### Foreign keys and joins

When joining tables, you connect a **foreign key** in one table to the **primary key** of another. Understanding the cardinality on each side is critical:

* **One-to-one**: each pupil has exactly one record in the demographics table
* **One-to-many**: each school has many pupils
* **Many-to-many**: pupils can attend multiple schools, and schools have multiple pupils (requires a linking table)

In [0]:
%sql
-- Demonstrate primary and foreign keys with two related tables

CREATE OR REPLACE TEMP VIEW schools AS
SELECT * FROM VALUES
  (100, 'Oak Academy',   'London',      'Academy'),
  (200, 'Elm School',    'Manchester',  'Maintained'),
  (300, 'Birch College', 'Birmingham',  'Academy')
AS t(school_urn, school_name, city, school_type);

CREATE OR REPLACE TEMP VIEW pupils AS
SELECT * FROM VALUES
  (1, 'Alice', '2012-03-15', 100),   -- school_urn is a FOREIGN KEY referencing schools
  (2, 'Bob',   '2011-07-22', 100),
  (3, 'Carol', '2012-11-01', 200),
  (4, 'Dan',   '2010-09-30', 300)
AS t(pupil_id, pupil_name, date_of_birth, school_urn);

-- Join using the foreign key relationship
SELECT
  p.pupil_id,
  p.pupil_name,
  p.school_urn,
  s.school_name,
  s.school_type
FROM pupils p
JOIN schools s ON p.school_urn = s.school_urn;

### Checking foreign key integrity

Joins only work correctly when the foreign key values in one table **actually exist** in the referenced table. When they don’t, you have **orphaned foreign keys** — rows that point to a record that isn’t there.

This can happen for many reasons:

* A school closed or was merged and removed from the reference table, but pupils still reference its URN
* A data load failed partway through, populating one table but not the other
* A manual data entry error introduced a URN that was never valid

Orphaned foreign keys are dangerous because they **silently drop rows** from inner joins. If you join pupils to schools and a pupil references a school that doesn’t exist, that pupil simply vanishes from your results — with no error or warning.

The pattern below uses a `LEFT JOIN` with a `WHERE ... IS NULL` filter to surface any orphaned records. In the example, pupil 5 (Eve) references school URN 999, which doesn’t appear in the schools table.

In [0]:
%sql
-- Add a pupil who references a school that doesn't exist in the schools table
CREATE OR REPLACE TEMP VIEW pupils AS
SELECT * FROM VALUES
  (1, 'Alice', '2012-03-15', 100),
  (2, 'Bob',   '2011-07-22', 100),
  (3, 'Carol', '2012-11-01', 200),
  (4, 'Dan',   '2010-09-30', 300),
  (5, 'Eve',   '2011-04-18', 999)    -- school_urn 999 does NOT exist in the schools table
AS t(pupil_id, pupil_name, date_of_birth, school_urn);

-- Find orphaned foreign keys: pupils referencing a school that doesn't exist
SELECT
  p.pupil_id,
  p.pupil_name,
  p.school_urn AS orphaned_school_urn
FROM pupils p
LEFT JOIN schools s ON p.school_urn = s.school_urn
WHERE s.school_urn IS NULL;

### Handling orphaned records — a business decision

Once you’ve identified orphaned foreign keys, you need to decide what to do with them. In some cases you may choose to **exclude these rows** from your analysis for data quality purposes — for example, if pupils reference schools that no longer exist in your reference data, including them could distort school-level aggregations or introduce misleading “unknown” categories.

However, removing data is never a purely technical decision. It is a **business decision** that should be:

* **Documented clearly** — record *what* was removed, *why*, and *how many rows* were affected
* **Proportionate** — if orphaned rows represent a significant share of the data, exclusion could introduce bias
* **Reversible** — filter rows out rather than deleting them, so the decision can be revisited
* **Communicated** — stakeholders should know that certain records were excluded and understand the impact

> **Good practice:** Add a comment or markdown cell to your notebook explaining the rationale whenever you filter out data. Future you (or a colleague) will thank you for it.